### Currency convertor Agent 

In [17]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model = 'ministral-3:3b')

In [34]:
from langchain_core.tools import tool
import requests

@tool 
def currency_convertor(amount: int, from_currency: str, to: str) -> dict:
    '''convert currency by using ISO 4217 Three Letter Currency Codes - e.g. USD for US Dollars, EUR for Euros, JPY for Japanese Yen etc'''

    result = requests.get(f"https://v6.exchangerate-api.com/v6/{'a3a9b4d36225c6a18ae6ff54'}/latest/{from_currency}")

    # print(result.json()["conversion_rates"][to] * amount)

    return result.json()["conversion_rates"][to] * amount

    

In [35]:
currency_convertor.invoke({"amount": 100, "from_currency": "USD", "to": "INR"})

9095.89

In [36]:
llm_with_tool = llm.bind_tools([currency_convertor])

In [37]:
from langchain_core.messages import HumanMessage


messages = []

query = HumanMessage(content="convert 100 USD to INR")

messages.append(query)


In [38]:
# Tool Calling LLM 

tool_calling = llm_with_tool.invoke(messages)

In [39]:
tool_calling

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T20:59:16.413707Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10736832500, 'load_duration': 6117060200, 'prompt_eval_count': 659, 'prompt_eval_duration': 3665856300, 'eval_count': 24, 'eval_duration': 876061500, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd34-3849-74c2-af3c-aa64b0349fef-0', tool_calls=[{'name': 'currency_convertor', 'args': {'amount': 100, 'from_currency': 'USD', 'to': 'INR'}, 'id': '6d843e45-412e-4a02-a040-02235a2f3dcc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 659, 'output_tokens': 24, 'total_tokens': 683})

In [40]:
messages.append(tool_calling)

In [41]:
tool_calling.tool_calls[0]

{'name': 'currency_convertor',
 'args': {'amount': 100, 'from_currency': 'USD', 'to': 'INR'},
 'id': '6d843e45-412e-4a02-a040-02235a2f3dcc',
 'type': 'tool_call'}

In [42]:
tool_execution_result  = currency_convertor.invoke(tool_calling.tool_calls[0])

In [43]:
tool_execution_result

ToolMessage(content='9095.89', name='currency_convertor', tool_call_id='6d843e45-412e-4a02-a040-02235a2f3dcc')

In [44]:
messages.append(tool_execution_result)

In [47]:
# now finally our messages list looks like this 
messages

[HumanMessage(content='convert 100 USD to INR', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T20:59:16.413707Z', 'done': True, 'done_reason': 'stop', 'total_duration': 10736832500, 'load_duration': 6117060200, 'prompt_eval_count': 659, 'prompt_eval_duration': 3665856300, 'eval_count': 24, 'eval_duration': 876061500, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd34-3849-74c2-af3c-aa64b0349fef-0', tool_calls=[{'name': 'currency_convertor', 'args': {'amount': 100, 'from_currency': 'USD', 'to': 'INR'}, 'id': '6d843e45-412e-4a02-a040-02235a2f3dcc', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 659, 'output_tokens': 24, 'total_tokens': 683}),
 ToolMessage(content='9095.89', name='currency_convertor', tool_call_id='6d843e45-412e-4a02-a040-02235a2f3dcc')]

In [45]:
final_response = llm_with_tool.invoke(messages)

In [46]:
final_response

AIMessage(content='100 USD is approximately **9095.89 INR** (as of the latest available data up to 2023-10-01, adjusted for recent exchange rate fluctuations). For the most accurate and up-to-date conversion, please verify with a recent exchange rate from a financial service.', additional_kwargs={}, response_metadata={'model': 'ministral-3:3b', 'created_at': '2026-01-20T20:59:51.9170397Z', 'done': True, 'done_reason': 'stop', 'total_duration': 3871946900, 'load_duration': 336281600, 'prompt_eval_count': 692, 'prompt_eval_duration': 982268200, 'eval_count': 68, 'eval_duration': 2358797000, 'logprobs': None, 'model_name': 'ministral-3:3b', 'model_provider': 'ollama'}, id='lc_run--019bdd34-ddc9-7c03-9a14-a46291289850-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 692, 'output_tokens': 68, 'total_tokens': 760})